# Retail Sales & Profitability Analysis — Python EDA
### Extending the MySQL + Power BI project with Python

This notebook extends the original **Retail Sales & Profitability Analysis** project
(originally built with MySQL for data extraction and Power BI for visualization)
by re-creating the core exploratory analysis in **Python (pandas, matplotlib, seaborn)**.

**Goal:** demonstrate the same business insights using a Python-based EDA workflow —
useful for analysts who need to move between SQL, BI tools, and Python notebooks
depending on the team and tooling.

**Dataset:** Retail Superstore transactions (9,994 rows) — same dataset used in the
original SQL/Power BI analysis.

**Original project:** SQL queries → Power BI dashboard (3 pages, 12+ visuals)
**This notebook:** pandas-based EDA → matplotlib/seaborn visuals → same business insights, reproduced independently in Python


## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style settings for clean, presentation-ready charts
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load dataset
# Replace with your actual file path / filename
df = pd.read_csv("Sample-Superstore.csv", encoding="latin1")

print(f"Shape: {df.shape}")
df.head()

## 2. Data Cleaning

Replicating the cleaning logic from the original SQL scripts (`01_clean_data.sql` etc.) in pandas.

In [ ]:
# Check data types and nulls
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Convert date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Derived columns used throughout the analysis
df['Order Month'] = df['Order Date'].dt.to_period('M')
df['Order Year'] = df['Order Date'].dt.year
df['Profit Margin %'] = (df['Profit'] / df['Sales']) * 100

# Sanity check
df[['Order Date', 'Order Month', 'Sales', 'Profit', 'Profit Margin %']].head()

## 3. Sales Overview

*Replicates: Sales Overview page of the Power BI dashboard*

In [ ]:
# Top-line KPIs (same as the Power BI KPI cards)
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
overall_margin = (total_profit / total_sales) * 100
total_orders = df['Order ID'].nunique()

print(f"Total Sales:        \${total_sales:,.2f}")
print(f"Total Profit:       \${total_profit:,.2f}")
print(f"Overall Margin:     {overall_margin:.2f}%")
print(f"Total Orders:       {total_orders:,}")

In [ ]:
# Monthly sales trend (seasonality)
monthly_sales = df.groupby('Order Month')['Sales'].sum()

fig, ax = plt.subplots()
monthly_sales.plot(kind='line', marker='o', ax=ax, color='#2c5f7c')
ax.set_title('Monthly Sales Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sales by Region and Category
region_cat = df.groupby(['Region', 'Category'])['Sales'].sum().unstack()

fig, ax = plt.subplots(figsize=(10, 6))
region_cat.plot(kind='bar', ax=ax, colormap='viridis')
ax.set_title('Sales by Region and Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=0)
plt.legend(title='Category')
plt.tight_layout()
plt.savefig('sales_by_region_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Profitability Analysis

*Replicates: Profitability Analysis page — discount vs profit relationship, loss-making sub-categories*

In [ ]:
# Discount vs Profit relationship (the key insight from the original project)
fig, ax = plt.subplots()
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, ax=ax)
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_title('Discount vs Profit by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Discount Rate')
ax.set_ylabel('Profit ($)')
plt.tight_layout()
plt.savefig('discount_vs_profit.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify the relationship
correlation = df['Discount'].corr(df['Profit'])
print(f"Correlation between Discount and Profit: {correlation:.3f}")
print("Confirms original SQL/Power BI finding: higher discounts are associated with lower profitability.")

In [ ]:
# Loss-making sub-categories (replicates the "Tables and Bookcases" finding)
subcat_profit = df.groupby('Sub-Category').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).sort_values('Total_Profit')

subcat_profit['Profit Margin %'] = (subcat_profit['Total_Profit'] / subcat_profit['Total_Sales']) * 100

print("Loss-making or lowest-margin sub-categories:")
subcat_profit.head(5)

In [ ]:
# Visualize profit margin by sub-category
fig, ax = plt.subplots(figsize=(10, 8))
subcat_sorted = subcat_profit.sort_values('Profit Margin %')
colors = ['#d62728' if x < 0 else '#2c5f7c' for x in subcat_sorted['Profit Margin %']]
ax.barh(subcat_sorted.index, subcat_sorted['Profit Margin %'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Profit Margin % by Sub-Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Profit Margin %')
plt.tight_layout()
plt.savefig('profit_margin_by_subcategory.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Regional Performance

*Replicates: regional performance gap finding (West vs Central)*

In [ ]:
regional_performance = df.groupby('Region').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique')
).sort_values('Total_Profit', ascending=False)

regional_performance['Profit Margin %'] = (regional_performance['Total_Profit'] / regional_performance['Total_Sales']) * 100
regional_performance

In [ ]:
fig, ax = plt.subplots()
regional_performance['Total_Profit'].plot(kind='bar', ax=ax, color=['#2c5f7c' if x == regional_performance['Total_Profit'].max() 
                                                                       else '#d62728' if x == regional_performance['Total_Profit'].min() 
                                                                       else '#7f9bb5' for x in regional_performance['Total_Profit']])
ax.set_title('Total Profit by Region', fontsize=14, fontweight='bold')
ax.set_ylabel('Profit ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('profit_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary of Findings

Reproducing the original SQL/Power BI analysis in Python confirms the same core insights:

- **Higher discount rates are associated with negative profitability**, particularly in Furniture and Technology categories — confirmed via correlation analysis (not just visual inspection).
- **Tables and Bookcases remain loss-making sub-categories** despite high sales volume, consistent with the original dashboard finding.
- **The West region leads in profit contribution**, while the Central region consistently underperforms.

### Why this matters
This notebook demonstrates the same analytical workflow in two different toolchains —
SQL/Power BI for stakeholder-facing dashboards, and Python for flexible, reproducible
exploratory analysis. Both arrive at the same business conclusions, which validates
the original findings independently.

### Next steps
- Build a simple regression model to quantify the discount-profit relationship further
- Automate this analysis as a reusable script for monthly reporting
